# Description

To load this from remote:
```
from transformers import pipeline

# It will download once and cache it for future use
ner_pipe = pipeline("ner", model="alexd063/bio-clinicalbert-finetuned-medmentions")
results = ner_pipe("Patient shows symptoms of acute appendicitis.")
print(results)
```

In [1]:
%%capture
!pip install transformers datasets evaluate seqeval accelerate torch numpy
!pip install --upgrade transformers

In [2]:
import os
# Suppress all tqdm/HuggingFace progress bars before any library is imported
os.environ["TQDM_DISABLE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"

In [3]:
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)
from huggingface_hub import notebook_login, whoami

In [4]:
import os
os.environ["TQDM_DISABLE"] = "1"

from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

from datasets import disable_progress_bars
disable_progress_bars()

from huggingface_hub.utils import disable_progress_bars as hub_disable_progress_bars
hub_disable_progress_bars()

In [5]:
PUSH_TO_HUB = True  # Set to False to skip login and hub push

if PUSH_TO_HUB:
    notebook_login()


In [6]:
class Config:
    MODEL_ID = "emilyalsentzer/Bio_ClinicalBERT"
    DATASET_ID = "Ben10x/MedMentions-MTI881-NER"
    HUB_REPO_ID = "alexd063/bio-clinicalbert-finetuned-medmentions"

    MAX_LENGTH = 512
    BATCH_SIZE = 16
    EPOCHS = 20
    LEARNING_RATE = 2e-5
    WARMUP_RATIO = 0.10
    WEIGHT_DECAY = 0.01

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = Config()


In [7]:
import time
notebook_start_time = time.time()


In [13]:
# ==========================================
# 1. Load Dataset & Extract Labels
# ==========================================
dataset = load_dataset("Ben10x/MedMentions-MTI881-NER")

# the dataset stores tags as strings (e.g., 'B-T062', 'I-T062', 'O').
# we need to extract all unique tags to create our label mappings.
unique_tags = set()
for split in dataset.keys():
    for row in dataset[split]["ner_tags"]:
        unique_tags.update(row)

label_list = sorted(list(unique_tags))
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

# ==========================================
# 2. Tokenization & Label Alignment
# ==========================================
# tokenization
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_ID)

def tokenize_and_align_labels(examples):
    # tokenize the pre-split words
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=config.MAX_LENGTH
    )

    labels = []
    for i, label_sequence in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # special tokens (like [CLS] and [SEP]) map to None. we set their label to -100 so they are ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # only label the first token of a given word.
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label_sequence[word_idx]])
            # for subsequent tokens of the same word, also assign -100 (or you can assign the I- tag).
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# apply to train, validation, and test splits
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)


In [14]:
%%capture
# capture widget output, github cannot show widgets.

# ==========================================
# 3. Metrics
# ==========================================

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # remove ignored index (special tokens)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": np.round(results["overall_precision"], 3),
        "recall": np.round(results["overall_recall"], 3),
        "f1": np.round(results["overall_f1"], 3),
        "accuracy": np.round(results["overall_accuracy"], 3),
    }

In [9]:

# ==========================================
# 4. Training
# ==========================================

training_args = TrainingArguments(
    output_dir="./bio-clinicalbert-medmentions",
    learning_rate=config.LEARNING_RATE,
    per_device_train_batch_size=config.BATCH_SIZE,
    per_device_eval_batch_size=config.BATCH_SIZE,
    num_train_epochs=config.EPOCHS,
    weight_decay=config.WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    hub_model_id=config.HUB_REPO_ID if PUSH_TO_HUB else None
)

model = AutoModelForTokenClassification.from_pretrained(
    config.MODEL_ID,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

# save the final model locally
trainer.save_model("./bio-clinicalbert-medmentions-final")


{'loss': '0.9208', 'grad_norm': '2.118', 'learning_rate': '1.966e-05', 'epoch': '0.3418'}
{'loss': '0.5775', 'grad_norm': '3.676', 'learning_rate': '1.932e-05', 'epoch': '0.6835'}


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': '0.5078', 'eval_precision': '0.512', 'eval_recall': '0.555', 'eval_f1': '0.532', 'eval_accuracy': '0.847', 'eval_runtime': '5.642', 'eval_samples_per_second': '518.3', 'eval_steps_per_second': '32.44', 'epoch': '1'}
{'loss': '0.5097', 'grad_norm': '2.112', 'learning_rate': '1.898e-05', 'epoch': '1.025'}
{'loss': '0.4401', 'grad_norm': '2.084', 'learning_rate': '1.863e-05', 'epoch': '1.367'}
{'loss': '0.4264', 'grad_norm': '2.317', 'learning_rate': '1.829e-05', 'epoch': '1.709'}


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': '0.4493', 'eval_precision': '0.547', 'eval_recall': '0.59', 'eval_f1': '0.568', 'eval_accuracy': '0.858', 'eval_runtime': '5.616', 'eval_samples_per_second': '520.6', 'eval_steps_per_second': '32.58', 'epoch': '2'}
{'loss': '0.4045', 'grad_norm': '1.836', 'learning_rate': '1.795e-05', 'epoch': '2.051'}
{'loss': '0.3464', 'grad_norm': '2.298', 'learning_rate': '1.761e-05', 'epoch': '2.392'}
{'loss': '0.3404', 'grad_norm': '2.3', 'learning_rate': '1.727e-05', 'epoch': '2.734'}


...

{'eval_loss': '0.4515', 'eval_precision': '0.61', 'eval_recall': '0.644', 'eval_f1': '0.627', 'eval_accuracy': '0.878', 'eval_runtime': '5.475', 'eval_samples_per_second': '534.1', 'eval_steps_per_second': '33.42', 'epoch': '6'}
{'loss': '0.1634', 'grad_norm': '1.636', 'learning_rate': '1.385e-05', 'epoch': '6.152'}
{'loss': '0.142', 'grad_norm': '3.119', 'learning_rate': '1.351e-05', 'epoch': '6.494'}
{'loss': '0.1453', 'grad_norm': '2.837', 'learning_rate': '1.317e-05', 'epoch': '6.835'}


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': '0.4708', 'eval_precision': '0.603', 'eval_recall': '0.647', 'eval_f1': '0.624', 'eval_accuracy': '0.877', 'eval_runtime': '5.506', 'eval_samples_per_second': '531.1', 'eval_steps_per_second': '33.24', 'epoch': '7'}
{'loss': '0.1303', 'grad_norm': '2.972', 'learning_rate': '1.282e-05', 'epoch': '7.177'}
{'loss': '0.116', 'grad_norm': '2.692', 'learning_rate': '1.248e-05', 'epoch': '7.519'}
{'loss': '0.1189', 'grad_norm': '2.73', 'learning_rate': '1.214e-05', 'epoch': '7.861'}


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': '0.5033', 'eval_precision': '0.609', 'eval_recall': '0.645', 'eval_f1': '0.626', 'eval_accuracy': '0.878', 'eval_runtime': '5.542', 'eval_samples_per_second': '527.6', 'eval_steps_per_second': '33.02', 'epoch': '8'}
{'train_runtime': '634.5', 'train_samples_per_second': '737.5', 'train_steps_per_second': '46.11', 'train_loss': '0.2977', 'epoch': '8'}


In [10]:
# ==========================================
# 5. Assessment on the Test Set
# ==========================================

test_results = trainer.evaluate(tokenized_datasets["test"])

print("--- Test Set Evaluation Results ---")
for key, value in test_results.items():
    # formatting key names for better readability (e.g., eval_f1 -> f1)
    metric_name = key.replace("eval_", "")
    print(f"{metric_name:10}: {value:.4f}")

# specific examples
# predictions, labels, metrics = trainer.predict(tokenized_datasets["test"])
# print("\nDetailed Test Metrics:", metrics)


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': '0.452', 'eval_precision': '0.612', 'eval_recall': '0.647', 'eval_f1': '0.629', 'eval_accuracy': '0.88', 'eval_runtime': '5.733', 'eval_samples_per_second': '510.4', 'eval_steps_per_second': '31.92', 'epoch': '8'}
--- Test Set Evaluation Results ---
loss      : 0.4520
precision : 0.6120
recall    : 0.6470
f1        : 0.6290
accuracy  : 0.8800
runtime   : 5.7327
samples_per_second: 510.4030
steps_per_second: 31.9220
epoch     : 8.0000


In [11]:
notebook_end_time = time.time()
elapsed = notebook_end_time - notebook_start_time
hours, rem = divmod(elapsed, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Total notebook execution time: {int(hours):02d}:{int(minutes):02d}:{int(seconds):02d}")


Total notebook execution time: 00:10:57


In [12]:

# ==========================================
# 7. Push to Hugging Face Hub
# ==========================================

if PUSH_TO_HUB:
    trainer.push_to_hub(
        commit_message="End of training",
        finetuned_from=config.MODEL_ID,
        dataset=config.DATASET_ID
    )
    tokenizer.push_to_hub(repo_id=config.HUB_REPO_ID)

    # sanity check: run inference from the remote model
    from transformers import pipeline

    ner_pipe = pipeline("ner", model=config.HUB_REPO_ID)
    results = ner_pipe("Patient shows symptoms of acute appendicitis.")
    print(results)


No files have been modified since last commit. Skipping to prevent empty commit.


[{'entity': 'B-T033', 'score': np.float32(0.9926732), 'index': 3, 'word': 'symptoms', 'start': 14, 'end': 22}, {'entity': 'B-T038', 'score': np.float32(0.99708194), 'index': 5, 'word': 'acute', 'start': 26, 'end': 31}, {'entity': 'I-T038', 'score': np.float32(0.99832183), 'index': 6, 'word': 'app', 'start': 32, 'end': 35}, {'entity': 'I-T038', 'score': np.float32(0.99675447), 'index': 7, 'word': '##end', 'start': 35, 'end': 38}, {'entity': 'I-T038', 'score': np.float32(0.99442), 'index': 8, 'word': '##icit', 'start': 38, 'end': 42}, {'entity': 'I-T038', 'score': np.float32(0.99552333), 'index': 9, 'word': '##is', 'start': 42, 'end': 44}]
